In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import nltk
import re
from nltk.tokenize import word_tokenize
from nltk.stem.wordnet import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.sentiment.vader import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')

pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth', None)

def display_():
    plt.tight_layout()
    plt.show()

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\panda\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


## Basic Cleaning
- Data import
- Select potential features of interest
- Rename columns

In [ ]:
DATA_DIR = Path("data")
REVIEWS_DF_PATH = DATA_DIR / "01_reviews_df.parquet"

reviews_df = pd.read_parquet(REVIEWS_DF_PATH)
df = reviews_df[['overall_rating','content']]
df = df.rename(columns={"overall_rating": "rating", "content": "reviews"})

## Text Cleaning Workflow:

1- Fix punctuation formatting before tokenization

2- Mask potential identifier/leakage terms (ratings/numbers, airline names). Store in separate column for additional feature engineering

3- Tokenize words, remove punctuation, force lowercase

4- Lemmatize tokens

5- Remove (customer determined) stopwords

In [3]:
def cleanText(text):
    cleaned_text = re.sub(r"[‘’]", "'", text)
    cleaned_text = re.sub(r"[“”]", '"', cleaned_text)
    return cleaned_text

# identifier/leakage mask functions are partially created with LLM assistance (Claude Sonnet 5.5)
rating_PH = " ratingXplaceholder "
def maskRatings(text):
    text = re.sub(r"\b\d{1,2}(?:\.\d)?\s*/\s*10\b", rating_PH, text, flags=re.I)
    text = re.sub(r"\b\d(?:\.\d)?\s*/\s*5\b", rating_PH, text, flags=re.I)
    text = re.sub(r"\b\w+ out of (?:10|5|ten|five)\b", rating_PH, text, flags=re.I)
    text = re.sub(r"\b(?:\d|one|two|three|four|five)[- ]?stars?\b(?![\s-]+hotels?\b)", rating_PH, text, flags=re.I)
    return text

airline_PH = " airlineXplaceholder "
NAMES = r"american|southwest|united|delta|spirit|frontier|hawaiian"
AFTER = r"states|kingdom"
def maskAirlines(text):
    text = re.sub(r"\balaska\s+air(?:\s?lines?)?\b(?:'s)?", airline_PH, text, flags=re.I)
    text = re.sub(rf"\b(?:{NAMES})\s+(?:air\s?lines?|airlines?|airways)\b(?:'s)?", airline_PH, text, flags=re.I)
    text = re.sub(r"\bjet\s?blue\b(?:'s)?", airline_PH, text, flags=re.I)
    text = re.sub(rf"\b(?:{NAMES})\b(?:'s)?(?!\s+(?:{AFTER})\b)", airline_PH, text, flags=re.I)
    return text

def maskNumbers(text):
    text = re.sub(r"[$£€]\s?\d[\d,]*(?:\.\d+)?", " moneyXplaceholder ", text)
    text = re.sub(r"\b\d[\d,.]*\b", " numberXplaceholder ", text)
    return text

##########################################################################################

token_remapper = {"ca": "can", "wo": "will", "n't": "not"}
def tokenize(text):
    token_list = [token_remapper.get(word, word) for word in word_tokenize(text.lower())]
    return [token for token in token_list if not re.fullmatch(r"\W+", token)]


lemmatizer = WordNetLemmatizer()
def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
def lemmatize(tokens):
    return [lemmatizer.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tag(tokens)]


stopwords = {"a","an","the","and","or","who","if","as","be","of","at","by","for","with","about",
             "have","this","my","we","between","into","through","during","to","from","in","it",
             "do","get","there","'s","that","our","i","'re","'ve","'ll","'m","'d"}
def removeStopwords(tokens):
    return [word for word in tokens if word not in stopwords]


# def fit_kept_words(reviews):
#     freq = Counter(word for review in reviews for word in review)
#     threshold = np.percentile(list(freq.values()), 75)
#     return {word for word, count in freq.items() if count > threshold}

# def remove_rare_words(reviews, kept_words):
#     return [' '.join(word for word in review if word in kept_words) for review in reviews]


df['reviews_masked'] = df['reviews'].map(cleanText).map(maskRatings).map(maskAirlines).map(maskNumbers)
df['reviews_cleaned'] = df['reviews_masked'].map(tokenize).map(lemmatize).map(removeStopwords).map(' '.join)

# view cleaning results
df[['reviews','reviews_masked','reviews_cleaned']].sample(5).head()

,reviews,reviews_masked,reviews_cleaned
20728,"Departure flight was on time, but that's where the good ends. My family of 4 flew from Milwaukee direct to Phoenix. Somehow only 2 of our 3 pieces of luggage arrived at Phoenix. 5 days later the missing suitcase still has not been located. Then my return flight got canceled 12 hours before departure with NO explanation as to why. I spent $1,427 for round trip tickets, bags, and upgraded seats. They claim I'll get a $488 credit for the canceled return flight but I haven't seen the credit yet. That's really bad math considering they only did half of the job. They never offered a solution and I had to scramble to get a return flight for $667 on a different airline to get home. Terrible communication and terrible customer service. I wish I had read some of the other reviews on this site before I booked with Spirit.","Departure flight was on time, but that's where the good ends. My family of numberXplaceholder flew from Milwaukee direct to Phoenix. Somehow only numberXplaceholder of our numberXplaceholder pieces of luggage arrived at Phoenix. numberXplaceholder days later the missing suitcase still has not been located. Then my return flight got canceled numberXplaceholder hours before departure with NO explanation as to why. I spent moneyXplaceholder for round trip tickets, bags, and upgraded seats. They claim I'll get a moneyXplaceholder credit for the canceled return flight but I haven't seen the credit yet. That's really bad math considering they only did half of the job. They never offered a solution and I had to scramble to get a return flight for moneyXplaceholder on a different airline to get home. Terrible communication and terrible customer service. I wish I had read some of the other reviews on this site before I booked with airlineXplaceholder .",departure flight on time but where good end family numberxplaceholder flew milwaukee direct phoenix somehow only numberxplaceholder numberxplaceholder piece luggage arrive phoenix numberxplaceholder day later miss suitcase still not locate then return flight cancel numberxplaceholder hour before departure no explanation why spent moneyxplaceholder round trip ticket bag upgraded seat they claim moneyxplaceholder credit canceled return flight but not see credit yet really bad math consider they only half job they never offer solution scramble return flight moneyxplaceholder on different airline home terrible communication terrible customer service wish read some other review on site before book airlinexplaceholder
9065,This is by far my worst flying experience in my life and I’ve been flying since the age of 4 which is 40 years ago and I’ve flown with many international airlines across the globe. This airline is a joke doesn’t even qualify to be a budget LLC. I flew with them 4 sectors all in first class including delta one. First leg was SEA-LAX first class in a very old 30+ years old B737 after boarding and take off and the pilot switch off the seat belt sign I asked the notorious cabin crew if I can use the toilet and I was the very first passenger to use what’s supposed to be the first class cabin toilet and it was filthy and floor full of water and when I informed the FA he said yes it’s flooding. Prior to my departure Delta sent me via their app to choose my meals and since I have specific dietary requirements I choose what’s suitable for me. When the FA was offering the meals he came to me and with a very low and unclear tone he offered me what options they have and when I told him I have a special meal request he said we don’t serve that on domestic flights so eventually I took the oat meal with some shea seeds which turned to be an awful non edible option. After landing and waiting for my luggage I found my suitcase was damaged beyond repair I went to Delta luggage service desk I spoke with the employee who was very unhelpful and not interested to be working she just did a form and gave it to me with a tracking number and ti

## Feature Engineering
1) Character count/review length
2) Count of words in all caps
3) Percent of words in all caps
4) Compounded VADER sentiment scores

In [4]:
def feature_engineering(df):
    df['char_count'] = df['reviews'].str.len()
    df['caps_count'] = df['reviews_masked'].str.count(r"\b[A-Z]{2,}\b")
    df['caps_pct'] = (df['caps_count'] / df['reviews_masked'].str.count(r"[A-Za-z]+(?:'[A-Za-z]+)?")).fillna(0) #pct words that are all caps

    sia = SentimentIntensityAnalyzer()
    df['sentiment'] = df['reviews_masked'].apply(lambda review: sia.polarity_scores(review)['compound'])
    return df

df = feature_engineering(df)

## Export for Modeling

In [6]:
df.to_parquet('data/02_cleaned_reviews_df.parquet')

----------